In [ ]:
!pip install -q ultralytics boxmot
!pip install -q --no-deps motmetrics xmltodict
import ultralytics, boxmot, motmetrics, numpy
print("ultralytics:", ultralytics.__version__)
print("boxmot:", boxmot.__version__)
print("motmetrics:", motmetrics.__version__)
print("numpy:", numpy.__version__)

In [ ]:
from pathlib import Path

print("=== VIDEOS ===")
vids = sorted(Path("/kaggle/input").rglob("*.mp4"))
for i, v in enumerate(vids):
    print(i, "|", v.name, "|", v.parts[3], f"| {v.stat().st_size/1e6:.0f} MB")

print("\n=== JSON ===")
for j in Path("/kaggle/input").rglob("*.json"):
    print(j, f"| {j.stat().st_size/1e6:.0f} MB")

print("\n=== MODEL ZIP ===")
for z in Path("/kaggle/input").rglob("*.zip"):
    print(z, f"| {z.stat().st_size/1e6:.0f} MB")

In [ ]:
import zipfile, os
from pathlib import Path

zip_path = Path("/kaggle/input/notebooks/payal1238gupta/ppe-3-0-v2/_output_.zip")
print("zip:", zip_path, "| exists:", zip_path.exists())

os.makedirs("/kaggle/working/phase3_output", exist_ok=True)
with zipfile.ZipFile(zip_path) as z:
    pts = [n for n in z.namelist() if n.endswith("phase3_best.pt")]
    print("mila:", pts)
    for n in pts:
        z.extract(n, "/kaggle/working/phase3_output")

MODEL_PATH = next(Path("/kaggle/working/phase3_output").rglob("phase3_best.pt"))
print("MODEL_PATH:", MODEL_PATH, f"| {MODEL_PATH.stat().st_size/1e6:.0f} MB")

In [ ]:
DEV_VIDEO  = [v for v in vids if "chunk_010" in v.name][0]
TEST_VIDEO = [v for v in vids if "chunk_032" in v.name][0]
GT_JSON    = Path("/kaggle/input/datasets/payal1238gupta/tracking-db-json-file/tracking_db.video_tracking (2).json")

print("DEV :", DEV_VIDEO.name)
print("TEST:", TEST_VIDEO.name)
print("GT  :", GT_JSON.exists())

In [ ]:
import cv2
from ultralytics import YOLO

person_model = YOLO("yolov8n.pt")

for tag, v in [("DEV", DEV_VIDEO), ("TEST", TEST_VIDEO)]:
    cap = cv2.VideoCapture(str(v))
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps = cap.get(cv2.CAP_PROP_FPS) or 10
    counts = []
    for i in range(1, 6):
        cap.set(cv2.CAP_PROP_POS_FRAMES, i * total // 6)
        ok, frame = cap.read()
        if not ok:
            counts.append(None); continue
        res = person_model(frame, conf=0.25, classes=[0], verbose=False)[0]
        counts.append(len(res.boxes))
    cap.release()
    print(f"{tag} | {v.name}")
    print(f"   {total} frames @ {fps:.1f} fps | persons at 5 frames = {counts}")

In [ ]:
import json
from collections import Counter

db = json.load(open(GT_JSON))
print("type:", type(db), "| len:", len(db))

first = db[0] if isinstance(db, list) else list(db.items())[0]
print("\n--- sample entry ---")
print(json.dumps(first, indent=2)[:1200])

In [ ]:
import time, json

def run_tracking(video_path, tag, tracker="bytetrack.yaml",
                 max_frames=None, write_frames=900, conf=0.25):
    cap = cv2.VideoCapture(str(video_path))
    fps = cap.get(cv2.CAP_PROP_FPS) or 10
    cap.release()

    stream = person_model.track(source=str(video_path), conf=conf, classes=[0],
                                tracker=tracker, persist=True,
                                stream=True, verbose=False)
    all_ids, per_frame, rows, out, n = set(), [], [], None, 0
    start = time.time()

    for res in stream:
        frame = res.orig_img
        if out is None and write_frames:
            h, w = frame.shape[:2]
            out = cv2.VideoWriter(f"track_{tag}.mp4",
                                  cv2.VideoWriter_fourcc(*"mp4v"), fps, (w, h))
        here = 0
        if res.boxes.id is not None:
            for box, tid, cf in zip(res.boxes.xyxy, res.boxes.id, res.boxes.conf):
                x1, y1, x2, y2 = map(int, box.tolist())
                tid = int(tid); all_ids.add(tid); here += 1
                rows.append([n, tid, x1, y1, x2 - x1, y2 - y1, float(cf)])
                if out is not None and n < write_frames:
                    cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
                    cv2.putText(frame, f"ID {tid}", (x1, y1 - 8),
                                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
        per_frame.append(here)
        if out is not None and n < write_frames:
            out.write(frame)
        n += 1
        if n % 2000 == 0:
            print(f"  {tag}: {n} frames | {n/(time.time()-start):.1f} fps | ids={len(all_ids)}")
        if max_frames and n >= max_frames:
            break

    if out is not None: out.release()
    with open(f"tracks_{tag}.txt", "w") as f:      # MOT format, metrics ke liye
        for fr, tid, x, y, w_, h_, cf in rows:
            f.write(f"{fr+1},{tid},{x},{y},{w_},{h_},{cf:.3f},-1,-1,-1\n")

    peak = max(per_frame) if per_frame else 0
    stats = {"video": Path(video_path).name, "tag": tag, "tracker": tracker,
             "frames": n, "proc_fps": round(n/(time.time()-start), 1),
             "unique_ids": len(all_ids), "peak_persons": peak,
             "avg_persons": round(sum(per_frame)/max(n, 1), 2),
             "fragmentation": round(len(all_ids)/max(peak, 1), 1)}
    print(stats)
    return stats

print("ready ✅")

In [ ]:
smoke_dev = run_tracking(DEV_VIDEO, "smoke_dev", max_frames=500, write_frames=0)

In [ ]:
base_dev  = run_tracking(DEV_VIDEO,  "base_dev")
base_test = run_tracking(TEST_VIDEO, "base_test")
json.dump([base_dev, base_test], open("baseline_stats.json", "w"), indent=2)

In [ ]:

from collections import Counter

gt_vids = Counter(e["video"] for e in db)
print("GT mein videos:", len(gt_vids))
for v, c in sorted(gt_vids.items()):
    print(f"  {v}  ->  {c} frames")

DEV_KEY  = DEV_VIDEO.stem
TEST_KEY = TEST_VIDEO.stem
print("\nDEV  in GT:", DEV_KEY in gt_vids, "|", DEV_KEY)
print("TEST in GT:", TEST_KEY in gt_vids, "|", TEST_KEY)

In [ ]:
for key, vid in [(DEV_KEY, DEV_VIDEO), (TEST_KEY, TEST_VIDEO)]:
    ent = [e for e in db if e["video"] == key]
    if not ent:
        print(key, "-> GT mein nahi\n"); continue
    frs = [e["frame_number"] for e in ent]
    ids = {t["track_id"] for e in ent for t in e.get("tracks", [])}
    cls = Counter(t["class"] for e in ent for t in e.get("tracks", []))
    cap = cv2.VideoCapture(str(vid)); tot = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)); cap.release()
    print(f"{key}")
    print(f"  GT frames: {min(frs)} se {max(frs)} | annotated entries: {len(ent)}")
    print(f"  video frames: {tot}")
    print(f"  unique GT track_ids: {len(ids)}")
    print(f"  classes: {dict(cls)}\n")

In [ ]:
import matplotlib.pyplot as plt

key, vid = DEV_KEY, DEV_VIDEO
ent = sorted([e for e in db if e["video"] == key], key=lambda e: e["frame_number"])
pick = ent[len(ent) // 2]
fno = pick["frame_number"]

cap = cv2.VideoCapture(str(vid))
cap.set(cv2.CAP_PROP_POS_FRAMES, fno)        # 0-based maan ke
ok, frame = cap.read(); cap.release()

for t in pick["tracks"]:
    x1, y1, x2, y2 = map(int, t["bbox"])
    cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 0, 255), 3)
    cv2.putText(frame, f'GT {t["track_id"]} {t["class"]}', (x1, y1 - 10),
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 255), 2)

plt.figure(figsize=(14, 8))
plt.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)); plt.axis("off")
plt.title(f"GT overlay | frame {fno}"); plt.show()
print(pick["tracks"])

In [ ]:
import numpy as np
if not hasattr(np, "asfarray"):
    np.asfarray = lambda a, dtype=np.float64: np.asarray(a, dtype=dtype)
    print("patch lagaya ✅")

import motmetrics as mm
print("motmetrics:", mm.__version__)

In [ ]:
import motmetrics as mm
import numpy as np

def load_gt(key):
    d = {}
    for e in db:
        if e["video"] != key:
            continue
        f = e["frame_number"] + 1          # MOT 1-based
        rows = []
        for t in e.get("tracks", []):
            x1, y1, x2, y2 = t["bbox"]
            rows.append((t["track_id"], x1, y1, x2 - x1, y2 - y1))
        if rows:
            d[f] = rows
    return d

def load_pred(path):
    d = {}
    for line in open(path):
        p = line.strip().split(",")
        f, tid = int(p[0]), int(p[1])
        d.setdefault(f, []).append((tid, *map(float, p[2:6])))
    return d

gt_dev  = load_gt(DEV_KEY)
gt_test = load_gt(TEST_KEY)
print("GT frames -> dev:", len(gt_dev), "| test:", len(gt_test))

In [ ]:
def evaluate(gt, pred, name):
    acc = mm.MOTAccumulator(auto_id=False)
    for f in sorted(gt):
        g, p = gt[f], pred.get(f, [])
        gb = np.array([r[1:] for r in g], dtype=float).reshape(-1, 4)
        pb = np.array([r[1:] for r in p], dtype=float).reshape(-1, 4)
        dist = mm.distances.iou_matrix(gb, pb, max_iou=0.5)
        acc.update([r[0] for r in g], [r[0] for r in p], dist, frameid=f)

    mh = mm.metrics.create()
    return mh.compute(acc, name=name, metrics=[
        "num_frames", "num_unique_objects", "mota", "motp", "idf1",
        "num_switches", "num_fragmentations",
        "num_false_positives", "num_misses",
        "mostly_tracked", "partially_tracked", "mostly_lost"])

print("ready ✅")

In [ ]:
import pandas as pd

pred_dev  = load_pred("tracks_base_dev.txt")
pred_test = load_pred("tracks_base_test.txt")
print("pred frames -> dev:", len(pred_dev), "| test:", len(pred_test))

summary_dev  = evaluate(gt_dev,  pred_dev,  "ByteTrack-DEV")
summary_test = evaluate(gt_test, pred_test, "ByteTrack-TEST")

out = pd.concat([summary_dev, summary_test])
print(mm.io.render_summary(out,
                           formatters=mm.metrics.create().formatters,
                           namemap=mm.io.motchallenge_metric_names))
out.to_csv("metrics_baseline.csv")

In [ ]:
import numpy as np

for name, gt, pred in [("DEV", gt_dev, pred_dev), ("TEST", gt_test, pred_test)]:
    gc = [len(gt[f]) for f in sorted(gt)]
    pc = [len(pred.get(f, [])) for f in sorted(gt)]
    print(f"{name}: GT boxes/frame avg={np.mean(gc):.2f} | PRED boxes/frame avg={np.mean(pc):.2f}")
    print(f"   frames jahan pred > gt: {sum(p > g for g, p in zip(gc, pc))} / {len(gc)}\n")

In [ ]:
import matplotlib.pyplot as plt

frames = sorted(gt_dev)
picks = [frames[len(frames)//4], frames[len(frames)//2], frames[3*len(frames)//4]]

cap = cv2.VideoCapture(str(DEV_VIDEO))
fig, axes = plt.subplots(1, 3, figsize=(22, 7))
for ax, f in zip(axes, picks):
    cap.set(cv2.CAP_PROP_POS_FRAMES, f - 1)
    ok, img = cap.read()
    if not ok: continue
    for tid, x, y, w, h in gt_dev[f]:                      # GT laal
        cv2.rectangle(img, (int(x), int(y)), (int(x+w), int(y+h)), (0, 0, 255), 3)
        cv2.putText(img, f"GT{tid}", (int(x), int(y)-8), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0,0,255), 2)
    for tid, x, y, w, h in pred_dev.get(f, []):            # PRED hara
        cv2.rectangle(img, (int(x), int(y)), (int(x+w), int(y+h)), (0, 255, 0), 2)
        cv2.putText(img, f"P{tid}", (int(x), int(y+h)+20), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0,255,0), 2)
    ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB)); ax.axis("off"); ax.set_title(f"frame {f}")
cap.release()
plt.tight_layout(); plt.show()

In [ ]:
from collections import Counter

centers = Counter()
for f, boxes in pred_dev.items():
    for tid, x, y, w, h in boxes:
        centers[(int((x + w/2) // 50) * 50, int((y + h/2) // 50) * 50)] += 1

print("sabse zyada baar detect hui jagah (x, y) -> count:")
for c, n in centers.most_common(8):
    print(f"  {c} -> {n} frames")


life = Counter()
for f, boxes in pred_dev.items():
    for tid, *_ in boxes:
        life[tid] += 1
print("\nsabse lambe track (id -> frames):")
for t, n in life.most_common(8):
    print(f"  ID {t} -> {n} frames")

In [ ]:
import numpy as np


POSTER_BOX = [380, 450, 500, 610]    

def in_poster(x, y, w, h, box=POSTER_BOX, thr=0.6):
    px1, py1, px2, py2 = box
    ix1, iy1 = max(x, px1), max(y, py1)
    ix2, iy2 = min(x + w, px2), min(y + h, py2)
    inter = max(0, ix2 - ix1) * max(0, iy2 - iy1)
    return inter / max(w * h, 1) > thr

pred_dev_clean = {}
removed = 0
for f, boxes in pred_dev.items():
    keep = [b for b in boxes if not in_poster(*b[1:])]
    removed += len(boxes) - len(keep)
    if keep: pred_dev_clean[f] = keep
print("hataye gaye boxes:", removed)

summary_clean = evaluate(gt_dev, pred_dev_clean, "ByteTrack-DEV-nomask")
print(mm.io.render_summary(summary_clean,
                           formatters=mm.metrics.create().formatters,
                           namemap=mm.io.motchallenge_metric_names))

In [ ]:
import numpy as np

cand = []
for f, boxes in pred_dev.items():
    for tid, x, y, w, h in boxes:
        cx, cy = x + w/2, y + h/2
        if 450 <= cx <= 600 and 130 <= cy <= 280:
            cand.append([x, y, x + w, y + h])

cand = np.array(cand)
print("poster boxes:", len(cand))
POSTER_BOX = [float(np.percentile(cand[:, 0], 5)),
              float(np.percentile(cand[:, 1], 5)),
              float(np.percentile(cand[:, 2], 95)),
              float(np.percentile(cand[:, 3], 95))]
print("POSTER_BOX =", [round(v) for v in POSTER_BOX])

In [ ]:
import matplotlib.pyplot as plt
cap = cv2.VideoCapture(str(DEV_VIDEO))
cap.set(cv2.CAP_PROP_POS_FRAMES, 11152)
ok, img = cap.read(); cap.release()
x1, y1, x2, y2 = map(int, POSTER_BOX)
cv2.rectangle(img, (x1, y1), (x2, y2), (255, 0, 255), 3)
plt.figure(figsize=(14, 8)); plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB)); plt.axis("off"); plt.show()

In [ ]:
def in_poster(x, y, w, h, box, thr=0.5):
    px1, py1, px2, py2 = box
    ix1, iy1 = max(x, px1), max(y, py1)
    ix2, iy2 = min(x + w, px2), min(y + h, py2)
    inter = max(0, ix2 - ix1) * max(0, iy2 - iy1)
    return inter / max(w * h, 1) > thr

def mask_preds(pred, box):
    out, rm = {}, 0
    for f, boxes in pred.items():
        keep = [b for b in boxes if not in_poster(*b[1:], box=box)]
        rm += len(boxes) - len(keep)
        if keep: out[f] = keep
    print("hataye:", rm)
    return out

pred_dev_clean = mask_preds(pred_dev, POSTER_BOX)

summary_clean = evaluate(gt_dev, pred_dev_clean, "ByteTrack-DEV-masked")
print(mm.io.render_summary(summary_clean,
                           formatters=mm.metrics.create().formatters,
                           namemap=mm.io.motchallenge_metric_names))

In [ ]:

gt_lost = sum(1 for f in gt_dev for (tid, x, y, w, h) in gt_dev[f]
              if in_poster(x, y, w, h, POSTER_BOX))
gt_total = sum(len(v) for v in gt_dev.values())
print(f"GT boxes mask ke andar: {gt_lost} / {gt_total}  ({100*gt_lost/gt_total:.1f}%)")
print("POSTER_BOX =", [round(v) for v in POSTER_BOX])

In [ ]:
from boxmot import BoxMOT
import inspect
print(inspect.signature(BoxMOT.__init__))


tracker_reid = BoxMOT(
    detector="yolov8n.pt",
    reid="osnet_x0_25_msmt17.pt",
    tracker="botsort",
    classes=[0],
)
print("BoxMOT ready ✅")

In [ ]:
import yaml, os

cfg = {
    "tracker_type": "botsort",
    "track_high_thresh": 0.5, "track_low_thresh": 0.1, "new_track_thresh": 0.6,
    "track_buffer": 60, "match_thresh": 0.8,
    "gmc_method": "sparseOptFlow",
    "proximity_thresh": 0.5, "appearance_thresh": 0.25,
    "with_reid": True, "model": "auto",
}
with open("botsort_reid.yaml", "w") as f:
    yaml.dump(cfg, f)
print(open("botsort_reid.yaml").read())

In [ ]:
reid_dev = run_tracking(DEV_VIDEO, "reid_dev", tracker="botsort_reid.yaml")

In [ ]:
import hashlib
for f in ["tracks_base_dev.txt", "tracks_reid_dev.txt"]:
    h = hashlib.md5(open(f, "rb").read()).hexdigest()
    n = sum(1 for _ in open(f))
    print(f, "|", n, "lines |", h[:12])

In [ ]:
import ultralytics, yaml
from ultralytics.utils import ROOT
print("ultralytics:", ultralytics.__version__)

default = ROOT / "cfg/trackers/botsort.yaml"
print("\n--- ultralytics ka apna botsort.yaml ---")
print(open(default).read())

In [ ]:
import yaml
from ultralytics.utils import ROOT

cfg = yaml.safe_load(open(ROOT / "cfg/trackers/botsort.yaml"))
cfg["with_reid"] = True
cfg["model"] = "auto"          # Cell 22 ke output ke hisaab se badalna pad sakta hai
cfg["track_buffer"] = 60
cfg["appearance_thresh"] = 0.25
cfg["proximity_thresh"] = 0.5

with open("botsort_reid2.yaml", "w") as f:
    yaml.dump(cfg, f)
print(open("botsort_reid2.yaml").read())

In [ ]:
pred_reid = load_pred("tracks_reid_dev.txt")
pred_reid_clean = mask_preds(pred_reid, POSTER_BOX)
s = evaluate(gt_dev, pred_reid_clean, "BotSORT-noReID-masked")
print(mm.io.render_summary(s, formatters=mm.metrics.create().formatters,
                           namemap=mm.io.motchallenge_metric_names))

In [ ]:
from boxmot import BoxMOT, ReIDModel
import inspect

print("--- BoxMOT ---")
print(inspect.signature(BoxMOT.__init__))
print("\n--- ReIDModel ---")
print(inspect.signature(ReIDModel.__init__))
print(ReIDModel.__doc__)

# available recipes/models
for attr in ["RECIPES", "MODELS", "available", "list_models"]:
    if hasattr(BoxMOT, attr): print(attr, ":", getattr(BoxMOT, attr))
    if hasattr(ReIDModel, attr): print("ReID", attr, ":", getattr(ReIDModel, attr))

In [ ]:
bm = BoxMOT(
    detector="yolov8n.pt",
    reid="osnet_x0_25_msmt17.pt",
    tracker="botsort",
    classes=[0],
)
print("BoxMOT ready ✅")
print(type(bm))
print([m for m in dir(bm) if not m.startswith("_")])

In [ ]:
import inspect
print("--- track ---")
print(inspect.signature(bm.track))
print(bm.track.__doc__)

print("\n--- train ---")
print(inspect.signature(bm.train))
print((bm.train.__doc__ or "")[:800])

In [ ]:
for t in [0.5, 0.7, 0.85, 0.95]:
    lost = sum(1 for f in gt_dev for (i,x,y,w,h) in gt_dev[f]
               if in_poster(x,y,w,h,POSTER_BOX,thr=t))
    tot  = sum(len(v) for v in gt_dev.values())
    cl   = {f:[b for b in bs if not in_poster(*b[1:],box=POSTER_BOX,thr=t)]
            for f,bs in pred_dev.items()}
    cl   = {f:v for f,v in cl.items() if v}
    s    = evaluate(gt_dev, cl, f"thr={t}")
    print(f"thr={t} | GT lost {100*lost/tot:.1f}% "
          f"| MOTA {float(s['mota'].iloc[0])*100:.1f}% "
          f"| IDF1 {float(s['idf1'].iloc[0])*100:.1f}% "
          f"| IDs {int(s['num_switches'].iloc[0])}")

In [ ]:
import subprocess
subprocess.run(["ffmpeg","-y","-i",str(DEV_VIDEO),
                "-vf","select='between(n,800,3800)'",
                "-vsync","0","-an","/kaggle/working/clip_dev.mp4"],
               capture_output=True)
clip = Path("/kaggle/working/clip_dev.mp4")
print(clip.exists(), round(clip.stat().st_size/1e6, 1), "MB")

In [ ]:
import time, glob
t0 = time.time()
res = bm.track(source=str(clip), device="0", conf=0.25,
               save_txt=True, verbose=False)
print("time:", round(time.time()-t0), "sec")
print(type(res))

for p in glob.glob("runs/**/*.txt", recursive=True)[:5]:
    print(p)
    print(open(p).readline())

In [ ]:
MASK_THR = 0.95       

p = "runs/track/clip_dev/tracks.txt"
print("--- pehli 5 lines ---")
for i, l in enumerate(open(p)):
    print(l.strip())
    if i >= 4: break

import numpy as np
arr = np.loadtxt(p, delimiter=",")
print("\nshape:", arr.shape)
for c in range(arr.shape[1]):
    u = np.unique(arr[:, c])
    print(f"col {c}: min={u.min():.1f} max={u.max():.1f} unique={len(u)}")

In [ ]:
import time
t0 = time.time()
res_full = bm.track(source=str(DEV_VIDEO), device="0", conf=0.25,
                    save_txt=True, verbose=False)
print("time:", round((time.time()-t0)/60, 1), "min")
import glob
print(glob.glob("runs/track/**/*.txt", recursive=True))

In [ ]:
import pandas as pd

def clean(pred, thr=MASK_THR):
    o = {f:[b for b in bs if not in_poster(*b[1:], box=POSTER_BOX, thr=thr)]
         for f, bs in pred.items()}
    return {f:v for f,v in o.items() if v}

base_final = evaluate(gt_dev, clean(pred_dev), "ByteTrack (baseline)")
print(mm.io.render_summary(base_final,
      formatters=mm.metrics.create().formatters,
      namemap=mm.io.motchallenge_metric_names))

In [ ]:
import numpy as np

BM_TXT = "runs/track/Cam_12_20260222T000000Z_20260223T000000Z_10fps_chunk_010/tracks.txt"
arr = np.loadtxt(BM_TXT, delimiter=",")
print("shape:", arr.shape)
print("\npehli 3 lines:")
for i in range(3):
    print(arr[i])

for c in range(arr.shape[1]):
    u = np.unique(arr[:, c])
    print(f"col {c}: min={u.min():.2f} max={u.max():.2f} unique={len(u)}")

In [ ]:
FRAME_COL, ID_COL = 0, 1       
BOX_COLS = [2, 3, 4, 5]

pred_bm = {}
for row in arr:
    f = int(row[FRAME_COL]) + 1        
    x1, y1, x2, y2 = row[BOX_COLS]
    pred_bm.setdefault(f, []).append((int(row[ID_COL]), x1, y1, x2 - x1, y2 - y1))

print("frames:", len(pred_bm), "| unique ids:", len({t for v in pred_bm.values() for (t,*_) in v}))

import pandas as pd
comp = pd.concat([
    evaluate(gt_dev, clean(pred_dev), "ByteTrack (baseline)"),
    evaluate(gt_dev, clean(pred_bm),  "BotSORT + OSNet ReID"),
])
print(mm.io.render_summary(comp, formatters=mm.metrics.create().formatters,
                           namemap=mm.io.motchallenge_metric_names))
comp.to_csv("phase4_comparison.csv")

In [ ]:
FRAME_COL, ID_COL = 0, 1

pred_bm = {}
for row in arr:
    f = int(row[FRAME_COL]) + 1
    x, y, w, h = row[2], row[3], row[4], row[5]
    pred_bm.setdefault(f, []).append((int(row[ID_COL]), x, y, w, h))

print("frames:", len(pred_bm), "| unique ids:",
      len({t for v in pred_bm.values() for (t, *_) in v}))

import pandas as pd
comp = pd.concat([
    evaluate(gt_dev, clean(pred_dev), "ByteTrack (baseline)"),
    evaluate(gt_dev, clean(pred_bm),  "BotSORT + OSNet ReID"),
])
print(mm.io.render_summary(comp, formatters=mm.metrics.create().formatters,
                           namemap=mm.io.motchallenge_metric_names))
comp.to_csv("phase4_comparison.csv")

In [ ]:
import time
t0 = time.time()
bm.track(source=str(TEST_VIDEO), device="0", conf=0.25, save_txt=True, verbose=False)
print("time:", round(time.time()-t0), "sec")

import numpy as np, glob
p = [x for x in glob.glob("runs/track/**/*.txt", recursive=True) if "chunk_032" in x][0]
a = np.loadtxt(p, delimiter=",")
pred_bm_test = {}
for r in a:
    pred_bm_test.setdefault(int(r[0])+1, []).append((int(r[1]), r[2], r[3], r[4], r[5]))

import pandas as pd
full = pd.concat([
    evaluate(gt_dev,  clean(pred_dev),     "DEV  | ByteTrack"),
    evaluate(gt_dev,  clean(pred_bm),      "DEV  | BotSORT+ReID"),
    evaluate(gt_test, clean(pred_test),    "TEST | ByteTrack"),
    evaluate(gt_test, clean(pred_bm_test), "TEST | BotSORT+ReID"),
])
print(mm.io.render_summary(full, formatters=mm.metrics.create().formatters,
                           namemap=mm.io.motchallenge_metric_names))
full.to_csv("phase4_final_metrics.csv")

In [47]:
!ls -lah /kaggle/working/*.mp4

-rw-r--r-- 1 root root 29M Sep  4 19:30 /kaggle/working/clip_dev.mp4
-rw-r--r-- 1 root root 30M Sep  4 19:23 /kaggle/working/track_base_dev.mp4
-rw-r--r-- 1 root root 27M Sep  4 19:24 /kaggle/working/track_base_test.mp4
-rw-r--r-- 1 root root 30M Sep  4 19:29 /kaggle/working/track_reid_dev.mp4
